In [ ]:
import nltk
from konlpy.tag import Okt
import gensim
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import MeCab
import warnings
warnings.filterwarnings('ignore')

#### 전처리 대상 데이터 불러오기

In [ ]:
raw_data = pd.read_excel('../P1_Data/02.Pretreatment/민원사무처리기준표_predata.xlsx',header = 0, index_col=0) #엑셀데이터 형태상 header지정
dictionary_9 =  pd.read_excel('../P1_Data/02.Pretreatment/03_01_dictionary_9.xlsx',index_col=0)
data = raw_data.copy()

In [ ]:
data

In [ ]:
dictionary_9


#### 사용컬럼 추출

In [ ]:
use_col = data.loc[:, ['사무명','소관부처', '구비서류']]
use_col

### 1. 전처리 함수 > 한글추출 및 띄어쓰기제거 및 해당컬럼 null값 row 제거 함수 생성

#### 매개변수 
- df : 전처리를 진행할 데이터프레임명 
- column_name : 전처리를 진행할 컬럼명

In [ ]:
def preprocess_data(df, column_name):
    # 지정한 컬럼에서 정규식 표현의 한글만 추출
    df[column_name +'_전처리'] = df[column_name].apply(lambda x: re.sub('[^가-힣]', '', str(x)))

    # 지정한 컬럼에서 띄어쓰기 제거 
    df[column_name +'_전처리'] = df[column_name +'_전처리'].apply(lambda x: ''.join(x.split()))
    
    # 지정한 컬럼에서 녈값의 경우 해당 row 삭제 
    df = df[df[column_name].notnull()]

    return df

In [ ]:
use_col_result = preprocess_data(use_col,'구비서류') #함수호출
use_col_result

### 2. 서류 리스트 사전에 있는 단어를 기준으로 포함되는 row 추출 함수 

#### 매개변수 
- df_1 : 첫번째 데이터프레임  
- df_2 : 두번째 데이터프레임
- col_name_1 : 첫번째 데이터프레임에서 비교할 컬럼 이름  
- col_name_2 : 두번째 데이터프레임에서 비교할 컬럼 이름  
- code_col : 두번째 데이터프레임에서 코드명을 포함한 컬럼 이름  

In [ ]:
def extract_word(df_1, df_2, col_name_1, col_name_2):

    # df_1에서 df_2의 단어를 포함하는 row 추출
    df_extracted = df_1[df_1[col_name_1].str.contains('|'.join(df_2[col_name_2]))]

    # 추출된 row에서 해당 단어 추출하여 새로운 컬럼 생성
    df_extracted['extracted_word'] = df_extracted[col_name_1].apply(lambda x: [word for word in df_2[col_name_2] if word in x])

    # 추출된 단어 explode
    df_extracted = df_extracted.explode('extracted_word')

    # df_1에서 df_2의 단어를 포함하지 않는 row 추출
    df_not_extracted = df_1[~df_1[col_name_1].str.contains('|'.join(df_2[col_name_2]))]
    
    return df_extracted, df_not_extracted


In [ ]:
df_extracted, df_not_extracted = extract_word(use_col_result, dictionary_9, '구비서류_전처리','구비서류_okt적용후_한글추출')# extract_word 함수 호출
df_extracted

#### 서류리스트에 해당되지않는 row 결과저장

In [ ]:
df_not_extracted.to_excel('../P1_Data/02.Pretreatment/03_02_compl_not_extracted9.xlsx',index = False) #사전에 포함되지 않는 행

In [ ]:
df_not_extracted

### 3. 결과1) 구비서류 value_count 함수

#### 매개변수 
- df_1 : 추출된 row 데이터프레임
- df_2 : 서류 리스트 사전 데이터프레임
- code_col_1 : df_1 code 컬럼명
- code_col_2 : df_2 code 컬럼명
- doc_col_2   : df_2 구비서류 컬럼명

In [ ]:
def get_value_cnt(df1, code_col_1, ):

    # df1에서 code_col_1 컬럼의 값의 빈도수를 계산하여 새로운 데이터프레임으로 추출
    df1_counts = df1[code_col_1].value_counts().reset_index()
    df1_counts.columns = [code_col_1, '빈도수']
    return df1_counts


In [ ]:
result1_value_cnt = get_value_cnt(df_extracted, 'extracted_word') #함수호출
result1_value_cnt.to_excel('../P3_Result/01.CSV/03_02_compl_value_cnt9.xlsx',index = False) #사전에 포함되지 않는 행
result1_value_cnt

### 4. 결과2) 서류를 기준으로 해당되는 서비스를 추출하는 함수 

#### 매개변수
- df : 데이터프레임
- word_col : 서류명 컬럼
- code_col : 코드명 컬럼
- group_col : 그룹바이 대상 컬럼

In [ ]:
def get_groupby(df, word_col, group_col, group_col2):
    df_service = df.groupby([word_col, group_col2])[group_col].agg(list).reset_index()
    df_service = df_service.explode(group_col)
    return df_service


In [ ]:
result2_groupby = get_groupby(df_extracted, 'extracted_word', '사무명','소관부처')
result2_groupby['서비스수'] = result2_groupby.groupby(['extracted_word'])['extracted_word'].transform('count')
result2_groupby = result2_groupby.reindex(columns=['extracted_word','서비스수', '소관부처', '사무명'])
result2_groupby = result2_groupby.sort_values(by='서비스수', ascending=False)
result2_groupby = result2_groupby.drop_duplicates()
result2_groupby.to_excel('../P3_Result/01.CSV/03_02_compl_groupby9.xlsx',index = False)
result2_groupby

### 5. 결과3) 서류를 기준으로 해당되는 서비스를 추출하는 함수 

#### 무인민원발급기 목록 기준 서비스별 해당되는 서류 개수 파악 

In [ ]:
df_extracted_use_col = df_extracted.loc[:, ['사무명','소관부처','extracted_word']]
df_extracted_use_col

In [ ]:
df_grouped = df_extracted_use_col.groupby(['사무명', '소관부처'])['extracted_word'].agg(','.join).reset_index()
df_grouped

In [ ]:
# count 컬럼 추가
df_grouped['count'] = df_grouped['extracted_word'].str.split(',').apply(lambda x: len(x))
df_grouped

#### 무인민원 발급기서류 매핑개수별 해당 서비스 수

In [ ]:
# count값을 기준으로 새로운 컬럼을 생성하고, 각 숫자에 해당되는 서비스명의 개수를 세어서 저장
df_grouped['매핑된 서류개수'] = pd.cut(df_grouped['count'], bins=[0, 1, 2, 3, 4, float('inf')], labels=['[1개]', '[2개]', '[3개]', '[4개]', '[5개이상]'], include_lowest=True)
result = df_grouped.groupby('매핑된 서류개수')['사무명'].count()
result